# Once Upon a Mirror, notebook 5: UCB

In section 5 of the game, the mirror added an exploration bonus to every estimate: 1 / √n, where n is the number
of dots in the product's row. The fewer dots, the bigger the bonus. Each morning the mirror showed the product
with the highest estimate + bonus. Here you write UCB in Python, play it on the game's race season, and race it
against every mirror of the game.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one only
if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 25
minutes.

The products, the customers and their probabilities are invented for the game. Start with the two folded cells
below: the mirror's kit, then the checks.

In [ ]:
#@title The mirror's kit (run me first) { display-mode: "form" }
# The mirror's kit. Run this cell first; you do not need to read it.
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

def estimate(row):
    """The estimate of a product: the average of its 1s and 0s (taps / showings); None when it was never shown."""
    if len(row) == 0:
        return None
    return sum(row) / len(row)


def estimates(rows):
    """The estimate of every product on the board, in the order of the rows."""
    return [estimate(row) for row in rows]

def pick_highest(values, next_number):
    """The row with the highest value. When several rows share the highest value, one of them at random, with a
    number from next_number: the rule of the game, so the same stream gives the same choice. A number is drawn only
    when there is a tie."""
    best = max(values)
    tied = [k for k in range(len(values)) if abs(values[k] - best) <= 1e-9]
    if len(tied) == 1:
        return tied[0]
    return tied[int(next_number() * len(tied))]


# For notebook 5: sqrt(n) is the square root of n, so sqrt(9) is 3.0
from math import sqrt

In [ ]:
#@title The checks of notebook 5 (run me too) { display-mode: "form" }
# The checks of notebook 5, play, show_bonus and show_ucb_board. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N5_NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
_N5_EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']

# The tests of bonus and scores: the inputs, the expected answer, the call as you would type it, and why that is the
# answer. Then what the usual slips give on the same tests ("skip" for the empty row, "error" for a crash), each
# with its hint.
_N5_VALUE_TESTS = {
    "bonus": [
        (([0],), 1.0,
         'bonus([0])',
         "One dot: 1/√1 = 1, the vacuum's bonus after initialize"),
        (([1, 0],), 0.7071067811865475,
         'bonus([1, 0])',
         "Two dots: 1/√2, the mattress's bonus after morning 6"),
        (([1, 0, 0],), 0.5773502691896258,
         'bonus([1, 0, 0])',
         "Three dots: 1/√3, the mattress's bonus after morning 7"),
        (([0, 0, 0, 0],), 0.5,
         'bonus([0, 0, 0, 0])',
         'Four dots: 1/√4 = 1/2. The bonus counts the dots, tap or no tap'),
        (([0, 1, 1, 1, 0, 1, 1, 1, 0],), 0.3333333333333333,
         'bonus([0, 1, 1, 1, 0, 1, 1, 1, 0])',
         'Nine dots: 1/√9 = 1/3'),
        (([],), None,
         'bonus([])',
         'A product never shown has no bonus yet: None, where the game writes ?'),
        (([1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0],), 0.2773500981126146,
         "bonus on the mattress's row at the end of the race season",
         'Its 13 dots: 1/√13'),
        (([0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0],), 0.11322770341445956,
         "bonus on the vacuum's row at the end of the race season",
         'Its 78 dots: 1/√78'),
    ],
    "scores": [
        (([[0], [0], [0], [1], [0]],), [1.0, 1.0, 1.0, 2.0, 1.0],
         'scores([[0], [0], [0], [1], [0]])',
         "The race season at Cinderella's after initialize: one dot each, so every bonus is 1, and the mattress got the only tap"),
        (([[0], [0], [0], [1, 0], [0]],), [1.0, 1.0, 1.0, 1.2071067811865475, 1.0],
         'scores([[0], [0], [0], [1, 0], [0]])',
         'After morning 6: the mattress has two dots'),
        (([[0], [0], [0], [1, 0, 0], [0]],), [1.0, 1.0, 1.0, 0.9106836025229592, 1.0],
         'scores([[0], [0], [0], [1, 0, 0], [0]])',
         'After morning 7: four products tie at 1, the mattress has 0.33 + 0.58 = 0.91'),
        (([[1, 0, 0]],), [0.9106836025229592],
         'scores([[1, 0, 0]])',
         "A board with one row, the mattress's after morning 7"),
        (([[0, 1, 1, 1, 0, 1, 1, 1, 0], [0, 0], [0, 0], [1, 0, 0], [0, 0]],), [1.0, 0.7071067811865475, 0.7071067811865475, 0.9106836025229592, 0.7071067811865475],
         "scores on Cinderella's board after 18 mornings of the race season",
         'One value per row, estimate + bonus'),
        (([[0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1], [0], [1, 0, 0], [0]],), [1.0, 1.120790811898598, 1.0, 0.9106836025229592, 1.0],
         "scores on Rapunzel's board after 30 mornings of UCB in the race season",
         'One value per row, estimate + bonus'),
        (([[0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0], [0, 0, 0], [1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0]],), [0.7157918059785621, 0.5773502691896258, 0.5773502691896258, 0.6619654827279993, 0.5773502691896258],
         "scores on Cinderella's board at the end of the race season",
         'The vacuum: 0.6 + 0.11 = 0.71'),
    ],
}
_N5_SLIPS = {
    "bonus": [
        ('n1', [0.7071067811865475, 0.5773502691896258, 0.5, 0.4472135954999579, 0.31622776601683794, 'skip', 0.2672612419124244, 0.1125087900926024],
         'n is the number of dots in the row, len(row), with nothing added: a row with one dot has the bonus 1/√1 = 1.'),
        ('n-1', ['error', 1.0, 0.7071067811865475, 0.5773502691896258, 0.35355339059327373, 'skip', 0.2886751345948129, 0.11396057645963795],
         'n is the number of dots in the row, len(row), with nothing taken away: a row with one dot has the bonus 1/√1 = 1.'),
        ('inverse', [1.0, 0.5, 0.3333333333333333, 0.25, 0.1111111111111111, 'skip', 0.07692307692307693, 0.01282051282051282],
         'take the square root first: 1 / sqrt(showings). 1 / showings shrinks too fast: 4 dots give 0.25, and the bonus of 4 dots is 1/√4 = 0.5.'),
        ('root', [1.0, 1.4142135623730951, 1.7320508075688772, 2.0, 3.0, 'skip', 3.605551275463989, 8.831760866327848],
         'the bonus is 1 divided by the square root: 1 / sqrt(showings). sqrt(showings) grows with the dots, and the bonus shrinks.'),
        ('half', [0.5, 0.25, 0.16666666666666666, 0.125, 0.05555555555555555, 'skip', 0.038461538461538464, 0.00641025641025641],
         "Python reads 1 / showings ** 1/2 as (1 / showings) / 2. Use the kit's square root: 1 / sqrt(showings)."),
        ('square', [1.0, 0.25, 0.1111111111111111, 0.0625, 0.012345679012345678, 'skip', 0.005917159763313609, 0.0001643655489809336],
         'showings ** 2 is showings times showings. The square root is sqrt(showings): return 1 / sqrt(showings).'),
        ('taps', ['error', 1.0, 1.0, 'error', 0.4082482904638631, 'skip', 0.4472135954999579, 0.14586499149789456],
         'n counts every dot of the row, taps and no taps alike: showings = len(row). sum(row) counts only the taps.'),
        ('estimate', [0.0, 0.5, 0.3333333333333333, 0.0, 0.6666666666666666, 'skip', 0.38461538461538464, 0.6025641025641025],
         'that is the estimate, taps / showings. The bonus looks only at the number of dots: 1 / sqrt(showings).'),
        ('round', [1.0, 0.71, 0.58, 0.5, 0.33, 'skip', 0.28, 0.11],
         'keep every decimal: return 1 / sqrt(showings) as it comes, without round. show_ucb_board rounds the numbers it prints.'),
    ],
    "scores": [
        ('est', [[0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.5, 0.0], [0.0, 0.0, 0.0, 0.3333333333333333, 0.0], [0.3333333333333333], [0.6666666666666666, 0.0, 0.0, 0.3333333333333333, 0.0], [0.0, 0.9166666666666666, 0.0, 0.3333333333333333, 0.0], [0.6025641025641025, 0.0, 0.0, 0.38461538461538464, 0.0]],
         'these are the estimates alone. Add the bonus of the row: score = estimate(row) + bonus(row).'),
        ('bonus', [[1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 0.7071067811865475, 1.0], [1.0, 1.0, 1.0, 0.5773502691896258, 1.0], [0.5773502691896258], [0.3333333333333333, 0.7071067811865475, 0.7071067811865475, 0.5773502691896258, 0.7071067811865475], [1.0, 0.20412414523193154, 1.0, 0.5773502691896258, 1.0], [0.11322770341445956, 0.5773502691896258, 0.5773502691896258, 0.2773500981126146, 0.5773502691896258]],
         'these are the bonuses alone. Add the estimate of the row: score = estimate(row) + bonus(row).'),
        ('times', [[0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.35355339059327373, 0.0], [0.0, 0.0, 0.0, 0.19245008972987526, 0.0], [0.19245008972987526], [0.2222222222222222, 0.0, 0.0, 0.19245008972987526, 0.0], [0.0, 0.18711379979593723, 0.0, 0.19245008972987526, 0.0], [0.06822694949332819, 0.0, 0.0, 0.10667311465869792, 0.0]],
         'the estimate and the bonus are added, with a plus: score = estimate(row) + bonus(row).'),
        ('minus', [[-1.0, -1.0, -1.0, 0.0, -1.0], [-1.0, -1.0, -1.0, -0.20710678118654746, -1.0], [-1.0, -1.0, -1.0, -0.24401693585629253, -1.0], [-0.24401693585629253], [0.3333333333333333, -0.7071067811865475, -0.7071067811865475, -0.24401693585629253, -0.7071067811865475], [-1.0, 0.7125425214347351, -1.0, -0.24401693585629253, -1.0], [0.489336399149643, -0.5773502691896258, -0.5773502691896258, 0.10726528650277006, -0.5773502691896258]],
         'the bonus is added to the estimate: score = estimate(row) + bonus(row), with a plus.'),
        ('n1', [[0.7071067811865475, 0.7071067811865475, 0.7071067811865475, 1.7071067811865475, 0.7071067811865475], [0.7071067811865475, 0.7071067811865475, 0.7071067811865475, 1.0773502691896257, 0.7071067811865475], [0.7071067811865475, 0.7071067811865475, 0.7071067811865475, 0.8333333333333333, 0.7071067811865475], [0.8333333333333333], [0.9828944326835045, 0.5773502691896258, 0.5773502691896258, 0.8333333333333333, 0.5773502691896258], [0.7071067811865475, 1.1166666666666667, 0.7071067811865475, 0.8333333333333333, 0.7071067811865475], [0.7150728926567049, 0.5, 0.5, 0.6518766265278091, 0.5]],
         'the bonus of a row with n dots is 1/√n, with nothing added to n: your bonus from task A gives it, bonus(row).'),
        ('round', [[1.0, 1.0, 1.0, 2.0, 1.0], [1.0, 1.0, 1.0, 1.21, 1.0], [1.0, 1.0, 1.0, 0.91, 1.0], [0.91], [1.0, 0.71, 0.71, 0.91, 0.71], [1.0, 1.12, 1.0, 0.91, 1.0], [0.72, 0.58, 0.58, 0.66, 0.58]],
         'keep every decimal: score = estimate(row) + bonus(row), without round. show_ucb_board rounds the numbers it prints.'),
    ],
}

# Each test of ucb is a season (its probabilities, its seed and its mornings) and what the game's UCB does on it,
# morning by morning: "shown" holds the row shown on each morning and "dots" her answer, 1 a tap and 0 no tap; "ties"
# holds the rows that share the highest estimate + bonus on each morning; "slips" holds the rows that a usual mistake
# shows on each morning.
_N5_TESTS = [
    {'about': "the race season at Cinderella's", 'call': 'ucb(cinderella, season_moods(RACE), strategy_numbers(RACE, "ucb"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '0123433000000001240000000003333330030000003000000030000000000004123000000000000000000000000000000000', 'dots': '0001000111011100001111011001101101001110000011010001001011110000000111111001111111111100110011100000', 'ties': ['0', '1', '2', '3', '4', '3', '3', '0124', '0', '0', '0', '0', '0', '0', '0', '0124', '024', '04', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '3', '3', '3', '3', '3', '0', '0', '3', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '124', '12', '2', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0'], 'slips': {'p': '0123403012403000030002410300000300000003000000003412000000000003000000000000003000000000000000000300', 'n1': '0123433333300000000000000000000000000000000000000000000000000000024100000000000000000000000000000000', 'est': '0123433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333', 'bonus': '0123412340412300214303214421300324143120213042304134210240314120303124201343201404231120344013213204', 'first': '0123433000000000000000000012440000003340030000000000040033000030012000000000000000000000000000000000', 'last': '0123433421033333333344421033333333333333343333332100000000000000000400000000000000000000000000000000', 'once': '0123433333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333'}},
    {'about': "the race season at Rapunzel's", 'call': 'ucb(rapunzel, season_moods(RACE), strategy_numbers(RACE, "ucb"))', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 103993, 'mornings': 100, 'shown': '0123413111111111131111111111111111111111111124401111111111111111143111111111111111111111111111111111', 'dots': '0101010111111101101111111101111101101111001001000111011011111001000111111001111111111101111011110010', 'ties': ['0', '1', '2', '3', '4', '13', '3', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '3', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0124', '1', '024', '04', '4', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '34', '3', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1'], 'slips': {'p': '0123411011112341111011111111111111111111101111111111111111111111111111111111111111111111111111111134', 'n1': '0123413111111111111111111111111111113311111111111111111133111111111111111111111111111111111111111111', 'est': '0123411311111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'bonus': '0123412340412300214303214421300324143120213042304134210240314120303124201343201404231120344013213204', 'first': '0123413111111111131111111111111111111111110124411111111111111111134111111111111111111111111111111111', 'last': '0123431111111111131111111111111111111111114211111011111111111111111311111111111111111111111111111111', 'once': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}},
    {'about': "another season at Cinderella's", 'call': 'ucb(cinderella, season_moods(1), strategy_numbers(1, "ucb"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 1, 'mornings': 100, 'shown': '0123411234000000000000000000000000001000000000000000000000000100000000000003333333333243333333000000', 'dots': '0100000000111111111011111011010101000011111011110010011000000011010100100101110100010001101100111110', 'ties': ['0', '1', '2', '3', '4', '1', '1', '0234', '034', '04', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '234', '3', '3', '3', '3', '3', '3', '3', '3', '3', '24', '4', '3', '3', '3', '3', '3', '3', '3', '0', '0', '0', '0', '0', '0'], 'slips': {'p': '0123403021403000030002140300000300000003000000003124000000000003000000000000003000000000000000000300', 'n1': '0123411123400000000000000000000000000000000000000000000000000000000000000111111100000000000000000000', 'est': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'bonus': '0123421304024134310230124240311423034012031242104324310123042340120314120342143013240312043410213240', 'first': '0123411023411110000000000000000000000000000000000000000000000233333334000000000000000000000000000000', 'last': '0123411432000000000000000000000000001000000000000000000000000100000000000004444432000000000000000000', 'once': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}},
    {'about': "another season at Rapunzel's", 'call': 'ucb(rapunzel, season_moods(2), strategy_numbers(2, "ucb"))', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 2, 'mornings': 100, 'shown': '0123411111111111111111111111111111111400223111111111111201111111111111111111111111111111111111111111', 'dots': '0100011111111011110101111011011111110010100111100101100001001100111011100111110111111011111010111111', 'ties': ['0', '1', '2', '3', '4', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '01234', '1', '1', '1', '1', '1', '1', '1', '0234', '023', '0', '23', '2', '3', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '02', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1'], 'slips': {'p': '0123411011112431111011111111111111111111101111111111111111111111111111111111111111111111111111111123', 'n1': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'est': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'bonus': '0123414023201344310202413431023142030142430123410204312214303104204213432014230120431413201403214203', 'first': '0123411111111111111111111111100111111233441111111111111034411111111111111411111111111111111111111111', 'last': '0123411111111111111111111111143211111011111111111111111111111111111111111111111111111111111111111111', 'once': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}},
    {'about': "a season of the collection Ready for the ball from section 3, at Cinderella's", 'call': 'ucb([0.6, 0.9, 0.6, 0.95, 0.6], season_moods(3), strategy_numbers(3, "ucb"))', 'p': [0.6, 0.9, 0.6, 0.95, 0.6], 'names': ['silk gloves', 'tiara', 'evening bag', 'ball gown', 'lipstick'], 'seed': 3, 'mornings': 100, 'shown': '0123423112313223113133113131313313113131321321323121313311333311113133131311311111111113211133333333', 'dots': '0111011111111101111111111111111111111111111111111101111110111011111111111111011111111100011011111111', 'ties': ['0', '1', '2', '3', '4', '123', '13', '1', '123', '23', '3', '123', '23', '2', '123', '13', '1', '13', '3', '13', '3', '13', '1', '13', '3', '13', '3', '13', '3', '13', '3', '13', '1', '13', '1', '13', '3', '13', '3', '13', '3', '2', '13', '3', '2', '13', '3', '2', '13', '1', '2', '13', '3', '13', '3', '13', '1', '13', '3', '3', '3', '3', '1', '1', '1', '13', '3', '13', '3', '13', '1', '13', '1', '13', '1', '13', '3', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '3', '2', '1', '1', '1', '3', '3', '3', '3', '3', '3', '3', '3'], 'slips': {'p': '0123431312403131331310243313133133131331330421331331313313313331331331331331333133401233133313313331', 'n1': '0123423112313223113133113131313313113131313133113133122131333311113313131133133333333331311133333333', 'est': '0123423121323113111123211113223333333333333333333333333333333333333333333333333333333333311133333333', 'bonus': '0123434012042311240301243410231042323014134201420334120413023142010324031421023410432431201230404231', 'first': '0123412312312313131313131313333332323233333131313313131313111133333333333333311111111113211133333333', 'last': '0123432132132131313131313131313131313132131313131313131313111133331313131313133333333331311133333333', 'once': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222'}},
    {'about': "a season of the collection Long hair from section 3, at Rapunzel's", 'call': 'ucb([0.02, 0.02, 0.95, 0.02, 0.02], season_moods(4), strategy_numbers(4, "ucb"))', 'p': [0.02, 0.02, 0.95, 0.02, 0.02], 'names': ['scissors', 'cap', 'ribbon', 'razor', 'haircut voucher'], 'seed': 4, 'mornings': 100, 'shown': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222224013222222222222', 'dots': '0010011011111111111101111111111111111111111011111111011111011111111111110111110110100000111111111111', 'ties': ['0', '1', '2', '3', '4', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '0134', '013', '13', '3', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2', '2'], 'slips': {'p': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'n1': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'est': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222', 'bonus': '0123440132312043401243201231402140314230230142130441203031424310231402341024012341023430213024140321', 'first': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222220134222222222222', 'last': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222224310222222222222', 'once': '0123422222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222222'}},
    {'about': "another season at Cinderella's", 'call': 'ucb(cinderella, season_moods(5), strategy_numbers(5, "ucb"))', 'p': [0.6, 0.2, 0.2, 0.4, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 5, 'mornings': 100, 'shown': '0123444120030444041324440000000000000000000000000000000000000000000000000000000000000000000000000000', 'dots': '0000100001000100000001001110110110111000100101100101101101011010110110100110010011111101110011101001', 'ties': ['0', '1', '2', '3', '4', '4', '4', '0123', '023', '03', '0', '3', '04', '4', '4', '4', '0', '4', '123', '23', '2', '4', '4', '4', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0'], 'slips': {'p': '0123403014203000030001240300000300000003000000003142000000000003000000000000003000000000000000000300', 'n1': '0123444412034444410324444222233333333333333333333333333333333333333333333333333333333333333333333333', 'est': '0123444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444', 'bonus': '0123413024132404103231042321402301441023142034023140123410231023430124321044130242301130244032114023', 'first': '0123444001230444041233333333333333333333334333000333333333333333333333333333330433333333333333333333', 'last': '0123444321044333333321110000000000000000000000000000000000000000000000000000000030000000000000000000', 'once': '0123444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444444'}},
    {'about': "another season at Rapunzel's", 'call': 'ucb(rapunzel, season_moods(6), strategy_numbers(6, "ucb"))', 'p': [0.4, 0.8, 0.2, 0.2, 0.2], 'names': ['robot vacuum', 'rope ladder', 'leaf blower', 'memory-foam mattress', 'satnav'], 'seed': 6, 'mornings': 100, 'shown': '0123411111142222031111211211143011111111111111111111111111111111111111111111111111111111111111111111', 'dots': '0100011001001100001100000010000011101110101110011111111111111011111111101111101111110111111011111111', 'ties': ['0', '1', '2', '3', '4', '1', '1', '1', '1', '1', '1', '0234', '023', '2', '2', '023', '03', '3', '1', '1', '1', '1', '2', '1', '1', '2', '1', '1', '1', '034', '03', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1'], 'slips': {'p': '0123411011114231111011111111111111111111101111111111111111111111111111111111111111111111111111111132', 'n1': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'est': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111', 'bonus': '0123441203432014213003142134024320141320402130214310234104232130412034201434301234021023143402132014', 'first': '0123411111100000234110111011101111111111111111111111111111111111111111111111111111111111111111111111', 'last': '0123411111143333201111311311142011111111111111111111111111111111111111111111111111111111111111111111', 'once': '0123411111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111'}},
]
_N5_SLIP_HINTS = {'p': 'the mirror does not know p: only tap reads it. It adds the bonus to the estimate on its board, and scores(rows) gives that for every row: product = pick_highest(scores(rows), next_number).', 'n1': 'your mirror uses the bonus of n + 1 dots. n is the number of dots in the row, len(row), with nothing added: scores(rows), with your bonus from task A, gives estimate + bonus.', 'est': 'your mirror shows the highest estimate and forgets the bonus, so a product with no tap at its first showing never comes back. Use estimate + bonus: product = pick_highest(scores(rows), next_number).', 'bonus': 'your mirror shows the highest bonus and forgets the estimate, so it only takes turns. Use estimate + bonus: product = pick_highest(scores(rows), next_number).', 'first': 'a tie is broken at random with a number from next_number, as in the game, and pick_highest does exactly that: product = pick_highest(scores(rows), next_number).', 'once': 'your mirror picked once, after initialize, and kept that product all season. UCB picks again every morning, because each new dot changes the estimate and the bonus of its row: the line with pick_highest goes inside the loop.', 'last': 'a tie is broken at random with a number from next_number, as in the game, and pick_highest does exactly that: product = pick_highest(scores(rows), next_number).'}
_N5_SUCCESS = {'bonus': 'bonus works on 8 rows: 1 dot gives 1, 2 dots 0.71, 3 dots 0.58, 4 dots 0.5, and a product never shown gives None.', 'scores': "scores works on 7 boards. On Cinderella's board after initialize it gives [1, 1, 1, 2, 1]: the mattress got the only tap.", 'ucb': "ucb works on 8 seasons, morning by morning as in the game, at Cinderella's and Rapunzel's and on the two collections of section 3. On Cinderella's race season it gets 52 taps in 100 mornings, a rate of 0.52: the game's numbers."}
_N5_TASKS = {"bonus": "task A", "scores": "task B", "ucb": "task C"}
_N5_NEEDS = {"bonus": [], "scores": ["bonus"], "ucb": ["bonus", "scores"]}
_N5_GAPS = {}
_N5_WORKING = {}
_N5_LOG = []
_N5_WORDS = {2: "Two", 3: "Three", 4: "Four", 5: "Five"}


def _n5_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n5_count(n, word):
    """1 dot, 3 dots"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _n5_ordinal(n):
    """1st, 2nd, 3rd, 4th, 11th, 21st"""
    if n % 100 in (11, 12, 13):
        return "%dth" % n
    return "%d%s" % (n, {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th"))


def _n5_morning(t):
    """morning 7 (the 8th of the season): Python counts the mornings from 0, the game from 1."""
    return "morning %d (the %s of the season)" % (t, _n5_ordinal(t + 1))


def _n5_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _n5_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _n5_dec(x):
    """At most two decimals, as on the board: 0.6, 0.71, 1."""
    text = "%.2f" % x
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    return text


def _n5_hund(x):
    """A number in hundredths, as the game's board shows it."""
    return int(x * 100 + 0.5) / 100


def _n5_number(x):
    """A number as the messages write it: two decimals, four when two would hide a difference."""
    try:
        if abs(x - round(x, 2)) < 1e-12:
            return _n5_dec(x)
        text = "%.4f" % x
    except Exception:
        return _n5_short(x)
    return text.rstrip("0").rstrip(".")


def _n5_text(value):
    """A value as the messages write it: a number with few decimals, a short list of them, None."""
    if _n5_is_number(value):
        return _n5_number(value)
    if isinstance(value, list) and len(value) <= 12:
        parts = []
        for x in value:
            parts.append(_n5_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n5_short(value)


def _n5_product(test, k):
    """the rope ladder (row 1, p = 0.2)"""
    if not isinstance(k, int) or k < 0 or k >= len(test["p"]):
        return "row %s" % _n5_short(k)
    return "the %s (row %d, p = %s)" % (test["names"][k], k, test["p"][k])


def _n5_label(k):
    """the 🤖 Robot vacuum (row 0)"""
    if isinstance(k, int) and 0 <= k < len(_N5_NAMES):
        return "the %s %s (row %d)" % (_N5_EMOJI[k], _N5_NAMES[k], k)
    return "row %s" % _n5_short(k)


def _n5_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N5_GAPS:
        return _N5_GAPS[code]
    # A gap such as "x = ..." stays in the compiled function; a gap on a line of its own is only visible in the
    # source, which Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N5_GAPS[code] = gap
    return gap


def _n5_is_board(value, rows=None):
    """True when value is a board: a list of lists of 1s and 0s (and rows lists, when rows is given)."""
    if not isinstance(value, list) or len(value) == 0:
        return False
    if rows is not None and len(value) != rows:
        return False
    for row in value:
        if not isinstance(row, list):
            return False
        for x in row:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return False
    return True


def _n5_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n5_same(answer, expected):
    """True when answer is the expected value: numbers up to 1e-9, None only for None, lists entry by entry."""
    if expected is None:
        return answer is None
    if _n5_is_number(expected):
        return _n5_is_number(answer) and abs(answer - expected) <= 1e-9
    if isinstance(expected, list):
        if not isinstance(answer, list) or len(answer) != len(expected):
            return False
        for i in range(len(expected)):
            if not _n5_same(answer[i], expected[i]):
                return False
        return True
    return answer == expected


def _n5_call(fn, inputs):
    """Run a function on fresh copies of the inputs, with its prints silenced. Gives back (answer, error); error is
    None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


# The check follows your mirror morning by morning. It hands your function moods and probabilities that write down
# every comparison of a mood with a probability, which is what tap does, so it knows which product your mirror showed
# on which morning, whatever the shape of your code. A sum with a probability is written down too.

def _n5_note(prob, mood, swapped):
    """One comparison: ("tap", row, morning, swapped, value) for a mood and a probability (row None and the value
    itself when it is not one of p), ("p", ...) for two probabilities."""
    if isinstance(mood, _n5_Mood):
        row = None
        if isinstance(prob, _n5_Prob):
            row = prob.row
        _N5_LOG.append(("tap", row, mood.morning, swapped, prob))
    elif isinstance(prob, _n5_Prob) and isinstance(mood, _n5_Prob):
        _N5_LOG.append(("p", None, None, False, None))


class _n5_Mood(float):
    """A mood that knows its morning: mood < p is her answer, the right way round."""

    def __lt__(self, other):
        _n5_note(other, self, False)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n5_note(other, self, False)
        return float.__le__(self, other)

    def __gt__(self, other):
        _n5_note(other, self, True)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n5_note(other, self, True)
        return float.__ge__(self, other)


class _n5_Prob(float):
    """A probability that knows its row: p > mood is her answer, the right way round. A sum with it is written down."""

    def __gt__(self, other):
        _n5_note(self, other, False)
        return float.__gt__(self, other)

    def __ge__(self, other):
        _n5_note(self, other, False)
        return float.__ge__(self, other)

    def __lt__(self, other):
        _n5_note(self, other, True)
        return float.__lt__(self, other)

    def __le__(self, other):
        _n5_note(self, other, True)
        return float.__le__(self, other)

    def __add__(self, other):
        _N5_LOG.append(("sum", None, None, False, None))
        return float.__add__(self, other)

    def __radd__(self, other):
        _N5_LOG.append(("sum", None, None, False, None))
        return float.__radd__(self, other)


def _n5_inputs(test):
    """Fresh inputs for a test, [p, moods, next_number]: moods and probabilities that write down each comparison, and
    a next_number that writes down each number it gives."""
    moods = season_moods(test["seed"])[:test["mornings"]]
    p = []
    for k in range(len(test["p"])):
        x = _n5_Prob(test["p"][k])
        x.row = k
        p.append(x)
    marked = []
    for t in range(len(moods)):
        x = _n5_Mood(moods[t])
        x.morning = t
        marked.append(x)
    numbers = strategy_numbers(test["seed"], "ucb")

    def next_number():
        _N5_LOG.append(("draw", None, None, False, None))
        return numbers()
    return [p, marked, next_number]


def _n5_run(fn, inputs):
    """Run fn on the inputs with its prints silenced and read what it did: answer, error (None when the call went
    through) and line (the line of your code where it happened); events, one (row, morning, swapped, value) per
    answer of the customer, in order (value is what was compared with the mood); draws, the number of answers before
    each number drawn from next_number; p_read, True when it compared two probabilities; p_sum, True when it added
    something to a probability. Never raises."""
    import contextlib, io, traceback
    del _N5_LOG[:]
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*inputs)
    except Exception as error:
        # The check's moods and probabilities are floats that write down each comparison: name them float
        run["error"] = ("%s: %s" % (type(error).__name__, error)).replace("_n5_Mood", "float").replace("_n5_Prob",
                                                                                                    "float")
        try:
            skip = (_n5_run.__code__.co_filename, tap.__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    events, draws, p_read, p_sum = [], [], False, False
    for kind, row, morning, swapped, value in _N5_LOG:
        if kind == "tap":
            events.append((row, morning, swapped, value))
        elif kind == "draw":
            draws.append(len(events))
        elif kind == "sum":
            p_sum = True
        else:
            p_read = True
    del _N5_LOG[:]
    run["events"], run["draws"], run["p_read"], run["p_sum"] = events, draws, p_read, p_sum
    return run


def _n5_board_of(test, upto):
    """The game's board after the first upto mornings of a test."""
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for t in range(upto):
        rows[int(test["shown"][t])].append(int(test["dots"][t]))
    return rows


def _n5_first_failure(name, fn):
    """The first test that fn fails, with what it did, or None when fn passes them all."""
    if name != "ucb":
        for inputs, expected, call, why in _N5_VALUE_TESTS[name]:
            answer, error = _n5_call(fn, inputs)
            if error is not None or not _n5_same(answer, expected):
                return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                        "why": why}
        return None
    for test in _N5_TESTS:
        run = _n5_run(fn, _n5_inputs(test))
        expected = _n5_board_of(test, test["mornings"])
        same = False
        if run["error"] is None:
            try:
                same = _n5_is_board(run["answer"]) and run["answer"] == expected
            except Exception:
                same = False
        if not same:
            run["test"] = test
            run["expected"] = expected
            return run
    return None


def _n5_works(name):
    """True when the function called name exists, is finished and passes its tests. Remembered for each version of
    the function and of the tasks it may use, so play can ask on every season."""
    fn = globals().get(name)
    code = getattr(fn, "__code__", None)
    if not callable(fn) or code is None or _n5_has_gap(fn):
        return False
    key = [code]
    for other in _N5_NEEDS[name]:
        key.append(getattr(globals().get(other), "__code__", None))
    key = tuple(key)
    if key not in _N5_WORKING:
        _N5_WORKING[key] = _n5_first_failure(name, fn) is None
    return _N5_WORKING[key]


def _n5_scores_of(rows):
    """Estimate + bonus of every row from your scores (task B), or None while it does not work."""
    if not _n5_works("scores"):
        return None
    for row in rows:
        if len(row) == 0:
            return None
    values, error = _n5_call(globals()["scores"], (rows,))
    if error is not None or not isinstance(values, list) or len(values) != len(rows):
        return None
    return values


def _n5_sum_text(row, value):
    """0.33 + 0.58 = 0.91: the estimate and the bonus as the board shows them, and their sum."""
    e = _n5_hund(estimate(row))
    b = _n5_hund(value - estimate(row))
    return "%s + %s = %s" % (_n5_dec(e), _n5_dec(b), _n5_dec(e + b))


# The hints of bonus and scores

def _n5_matches(answers, values):
    """True when the answers on all the tests are those of one slip."""
    for i in range(len(values)):
        wanted, got = values[i], answers[i]
        if isinstance(wanted, str):
            if wanted == "skip":
                continue
            if not (isinstance(got, str) and got == "error"):
                return False
        elif not _n5_same(got, wanted):
            return False
    return True


def _n5_slip_hint(name, fn):
    """The hint of the usual slip whose answers are fn's on every test, or None."""
    answers = []
    for inputs, wanted, call, why in _N5_VALUE_TESTS[name]:
        got, problem = _n5_call(fn, inputs)
        if problem is not None:
            got = "error"
        answers.append(got)
    for slip, values, hint in _N5_SLIPS[name]:
        if _n5_matches(answers, values):
            return hint
    return None


def _n5_hint_bonus(fn, failure):
    row = failure["inputs"][0]
    answer, error, expected = failure["answer"], failure["error"], failure["expected"]
    if len(row) == 0:
        if error is not None:
            return ("an empty row has no dots, and 1 / sqrt(0) divides by 0. Keep the first two lines of the "
                    "function: for an empty row they return None.")
        return ("a product never shown has no bonus yet. Keep the first two lines of the function: for an empty row "
                "they return None.")
    if error is not None and error.startswith("NameError"):
        if "sqrt" in error:
            return "sqrt comes from the mirror's kit: run its folded cell at the top, then this cell and the check."
        if "math" in error:
            return "sqrt is ready to use as it is, without math. in front: 1 / sqrt(showings)."
        return "Python does not know a name in your function: check its spelling. Inside it you can use row, len, sqrt."
    hint = _n5_slip_hint("bonus", fn)
    if hint is not None:
        return hint
    if error is not None:
        if "TypeError" in error:
            return "sqrt takes one number, the showings: showings = len(row), then return 1 / sqrt(showings)."
        return "showings = len(row), then return 1 / sqrt(showings)."
    if answer is None:
        return "end the function with a return line: return 1 / sqrt(showings)."
    if not _n5_is_number(answer):
        return "return one number, 1 / sqrt(showings)."
    for places in (1, 2, 3):
        if abs(answer - round(expected, places)) < 1e-12:
            return ("keep every decimal: return 1 / sqrt(showings) as it comes, without round. show_ucb_board rounds "
                    "the numbers it prints.")
    return "the bonus of a row with n dots is 1 / √n: showings = len(row), then return 1 / sqrt(showings)."


def _n5_hint_scores(fn, failure):
    rows = failure["inputs"][0]
    answer, error, expected = failure["answer"], failure["error"], failure["expected"]
    if error is not None:
        if error.startswith("NameError"):
            if "bonus" in error:
                return "bonus is your function from task A: run its cell, then this one."
            return ("Python does not know a name in your function: check its spelling. Inside it you can use rows, "
                    "row, values, score, estimate and bonus.")
        if "TypeError" in error and "NoneType" in error:
            return ("estimate(row) and bonus(row) each give a number for a row with dots: score = estimate(row) + "
                    "bonus(row). Give them the row, a list of 1s and 0s.")
        if "TypeError" in error:
            return "estimate and bonus each take one row: score = estimate(row) + bonus(row). rows is the whole board."
    hint = _n5_slip_hint("scores", fn)
    if hint is not None:
        return hint
    if error is not None:
        return "in the loop: score = estimate(row) + bonus(row), then values.append(score)."
    if answer is None:
        return "end the function with return values, after the loop."
    if _n5_is_number(answer):
        return ("that is one number. Return the list values, with one estimate + bonus for each row of the board: "
                "return values, after the loop.")
    if not isinstance(answer, list):
        return "return the list values, with one estimate + bonus for each row of the board."
    if len(answer) == 0:
        return ("values is still empty: inside the loop, put each score at the end of the list with "
                "values.append(score).")
    for x in answer:
        if isinstance(x, (list, tuple)):
            return "each entry of values is one number, estimate + bonus. Append score, the number of the row."
    if len(answer) == 1 and len(expected) > 1:
        return ("the function stops after the first row: the line return values belongs after the loop, at the same "
                "indentation as for.")
    if len(answer) != len(expected):
        return "one score per row: append exactly once in each round of the loop."
    if len(answer) > 1 and answer.count(answer[0]) == len(answer) and expected.count(expected[0]) != len(expected):
        return "every score is the same: in each round of the loop, use the row of that round, estimate(row) + bonus(row)."
    for k in range(len(expected)):
        if not _n5_same(answer[k], expected[k]):
            return ("row %d should get %s and got %s. In each round of the loop, score = estimate(row) + bonus(row)."
                    % (k, _n5_text(expected[k]), _n5_text(answer[k])))
    return "in the loop: score = estimate(row) + bonus(row), then values.append(score)."


# The hints of ucb

def _n5_explain_error(run):
    """The hint for a mirror that stopped with an error."""
    error = run["error"]
    text = (run["line"] or "").replace(" ", "")
    test = run["test"]
    last = len(test["p"]) - 1
    if "positional argument" in error:
        return "keep the first line of the function as it was given: def ucb(p, moods, next_number):"
    if error.startswith("NameError"):
        if "scores" in error:
            return "scores is your function from task B: run its cell, then this one."
        if "bonus" in error:
            return "bonus is your function from task A: run its cell, then this one."
        if "random" in error or "randint" in error or "choice" in error:
            return ("a tie is broken with a number from next_number, as in the game: pick_highest(scores(rows), "
                    "next_number) does it for you.")
        return ("Python does not know a name in your function: check its spelling. Inside it you can use p, moods, "
                "next_number, rows, morning, product, dot, tap, scores and pick_highest.")
    if ("NoneType" in error and "+" in error) or error.startswith("ZeroDivisionError"):
        return ("a row with no dots has no estimate and no bonus yet (None), so estimate + bonus fails. Initialize "
                "first: on mornings 0 to len(p) - 1 show each product once, product = morning, so that every row has "
                "a dot before the first scores(rows).")
    if error.startswith("IndexError"):
        if "moods[" in text:
            return ("a morning went past the end of the season. The remaining mornings are range(len(p), len(moods)), "
                    "from the first morning after initialize to the last one.")
        if "p[" in text or "rows[" in text:
            return ("product is a row from 0 to %d: pick_highest(scores(rows), next_number) gives one, and on the "
                    "mornings of initialize product = morning." % last)
        return ("an index went past the end of a list. Python counts from 0: the rows are 0 to %d, the mornings 0 to "
                "%d." % (last, test["mornings"] - 1))
    if error.startswith("TypeError"):
        if "indices must be integers" in error and "float" in error:
            return ("product is a row number, such as 0 or 3. pick_highest(scores(rows), next_number) gives the row "
                    "with the highest estimate + bonus; max gives the score itself.")
        if "not callable" in error:
            return ("give pick_highest the stream itself, with no brackets after next_number: "
                    "pick_highest(scores(rows), next_number).")
        if "has no len" in error or "not iterable" in error:
            return "scores takes the board: scores(rows), the list of rows. p is for tap only."
        if "not supported between" in error:
            return ("tap compares two numbers: the probability of the product shown and the mood of the morning, "
                    "tap(p[product], moods[morning]).")
        if "NoneType" in error:
            return ("product must be a row number: product = pick_highest(scores(rows), next_number) after "
                    "initialize, and product = morning during it.")
    if error.startswith("AttributeError") and "append" in error:
        return ("append works on a list. rows[product] is the row of the product shown, a list, and "
                "rows[product].append(dot) adds the dot at its end.")
    if error.startswith("ValueError") and "empty" in error:
        return "pick_highest needs one value per row: pick_highest(scores(rows), next_number)."
    return ("initialize with product = morning, then on each remaining morning product = pick_highest(scores(rows), "
            "next_number), her answer with tap and the update with append.")


def _n5_shape(run):
    """(what, hint) when the mirror did not give a board with one row per product, or None."""
    test, answer = run["test"], run["answer"]
    rows = len(test["p"])
    if answer is None:
        return ("your function gave no answer (None)", "end the function with return rows, after the loop.")
    if not isinstance(answer, list):
        return ("your function gave %s, and a board is a list of rows" % _n5_short(answer),
                "return rows, the board: one list per product.")
    if len(answer) == 0:
        return ("your function gave an empty board, []",
                "the board starts with one empty row per product, as given in the first lines of the function.")
    for row in answer:
        if not isinstance(row, list):
            return ("your function gave %s, a board with numbers next to its rows" % _n5_short(answer),
                    "add each dot to the row of the product shown: rows[product].append(dot). rows.append(dot) puts "
                    "it next to the rows.")
    if len(answer) != rows:
        return ("your function gave a board of %s for %s" % (_n5_count(len(answer), "row"), _n5_count(rows, "product")),
                "one row per product: the board starts with one empty row for each probability in p.")
    for r in range(rows):
        for x in answer[r]:
            if not isinstance(x, (bool, int, float)) or x not in (0, 1):
                return ("your function gave a board whose row %d holds %s" % (r, _n5_short(x)),
                        "each dot is her answer, 1 or 0: dot = tap(p[product], moods[morning]).")
    return None


def _n5_answers(run):
    """(what, hint) when her answers did not reach the board as she gave them, or None. The check rebuilds the board
    from your own mirror's showings and compares it with the board it returned."""
    test, answer, events = run["test"], run["answer"], run["events"]
    for row, morning, swapped, value in events:
        if swapped:
            return ("on %s her answer is the wrong way round: she taps when her mood is below the probability"
                    % _n5_morning(morning),
                    "tap takes the probability first, then the mood: dot = tap(p[product], moods[morning]).")
    for row, morning, swapped, value in events:
        if row is None and isinstance(value, int) and not isinstance(value, bool):
            return ("on %s her answer came from %d, a row number in the place of a probability"
                    % (_n5_morning(morning), value),
                    "tap takes the probability of the product shown, p[product], then the mood: dot = tap(p[product], "
                    "moods[morning]).")
        if row is None:
            return ("on %s her answer came from a probability that is not one of p" % _n5_morning(morning),
                    "use p, the probabilities that the function gets: dot = tap(p[product], moods[morning]). The check "
                    "also tries the other customer and the collections of section 3.")
    for i in range(1, len(events)):
        if events[i][1] == events[i - 1][1]:
            return None             # two answers on one morning: the reading of the mornings says so
    moods = season_moods(test["seed"])
    rows = []
    for k in range(len(test["p"])):
        rows.append([])
    for row, morning, swapped, value in events:
        if isinstance(morning, int) and 0 <= morning < len(moods):
            rows[row].append(tap(test["p"][row], moods[morning]))
    if rows == answer:
        return None
    dots, want = 0, 0
    for k in range(len(rows)):
        dots = dots + len(answer[k])
        want = want + len(rows[k])
    if dots < want:
        return ("your board has %s for %s of the customer" % (_n5_count(dots, "dot"), _n5_count(want, "answer")),
                "update: add each answer to the row of the product shown, once every morning, after initialize too: "
                "rows[product].append(dot). Without it the estimates and the bonuses never change.")
    if dots > want:
        return ("your board has %s for %s of the customer" % (_n5_count(dots, "dot"), _n5_count(want, "answer")),
                "one dot per morning: append exactly once in each round of each loop.")
    for k in range(len(rows)):
        if len(answer[k]) != len(rows[k]):
            return ("the row of %s has %s, and your mirror showed it %s" % (_n5_product(test, k),
                                                                          _n5_count(len(answer[k]), "dot"),
                                                                          _n5_count(len(rows[k]), "time")),
                    "her answers are right, and some went into another row. Add each dot to the row of the product "
                    "shown: rows[product].append(dot).")
    return ("every row has the right number of dots, and some dots are not her answers",
            "add her answer itself: dot = tap(p[product], moods[morning]), then rows[product].append(dot).")


def _n5_too_short(test, j):
    """(what, hint) when the first j mornings are the game's and the mirror stops there."""
    k = len(test["p"])
    if j < k:
        return ("your mirror stopped during initialize, after %s" % _n5_count(j, "morning"),
                "initialize shows every product once: for morning in range(len(p)), with product = morning.")
    if j == k:
        return ("your mirror stopped after the %d mornings of initialize, and the season has %d" % (k,
                                                                                                 test["mornings"]),
                "the loop over the remaining mornings, for morning in range(len(p), len(moods)), shows the highest "
                "estimate + bonus on every morning after initialize.")
    return ("your mirror played %s of the %d" % (_n5_count(j, "morning"), test["mornings"]),
            "the remaining mornings run to the last morning of the season: for morning in range(len(p), len(moods)).")


def _n5_wrong_morning(test, events, j):
    """(what, hint) when the season's j-th answer came from the mood of another morning than morning j."""
    got = events[j][1]
    k = len(test["p"])
    if j > 0 and got == events[j - 1][1]:
        return ("your mirror asked for her answer on %s twice" % _n5_morning(got),
                "one answer per morning: call tap once in each round of the loop, and append its dot once.")
    if j == 0:
        return ("your mirror started with the mood of morning %d" % got,
                "Python counts the mornings from 0: range(len(p)) gives 0, 1, 2 and so on, and moods[0] is the mood "
                "of the first morning.")
    if j == k and got == 0:
        return ("after initialize your mirror started again from morning 0",
                "the remaining mornings start right after initialize: for morning in range(len(p), len(moods)).")
    if got > j:
        return ("your mirror skipped %s" % _n5_morning(j),
                "range(len(p), len(moods)) starts with morning len(p), the first morning after initialize, and ends "
                "with the last morning of the season.")
    return ("on %s her answer came from the mood of morning %d" % (_n5_morning(j), got),
            "each morning has its own mood: dot = tap(p[product], moods[morning]).")


def _n5_highest(test, j):
    """The rows that share the highest estimate + bonus before morning j, the game's pick, and the scores of the board
    from your scores (None while it does not work)."""
    tied = []
    for c in test["ties"][j]:
        tied.append(int(c))
    board = _n5_board_of(test, j)
    return tied, int(test["shown"][j]), board, _n5_scores_of(board)


def _n5_played(events):
    """The rows shown, one digit per morning, as in a test's "shown"."""
    played = ""
    for e in events:
        played = played + str(e[0])
    return played


def _n5_which_slip(fn, test, events):
    """The usual slip that plays the seasons your mirror plays, or None. Two slips can play the same season, so the
    other seasons of the check decide between them."""
    candidates = []
    for slip in ("p", "n1", "est", "bonus", "first", "last", "once"):
        if test["slips"][slip] == _n5_played(events):
            candidates.append(slip)
    for other in _N5_TESTS:
        if len(candidates) <= 1:
            break
        if other is test:
            continue
        run = _n5_run(fn, _n5_inputs(other))
        if run["error"] is not None:
            continue
        kept = []
        for slip in candidates:
            if other["slips"][slip] == _n5_played(run["events"]):
                kept.append(slip)
        if len(kept) > 0:
            candidates = kept
    if len(candidates) == 0:
        return None
    return candidates[0]


def _n5_wrong_choice(fn, test, run, j):
    """(what, hint) when on morning j, after initialize, the mirror showed another product than the game's."""
    events = run["events"]
    row = events[j][0]
    tied, right, board, values = _n5_highest(test, j)
    what = "on %s your mirror showed %s" % (_n5_morning(j), _n5_product(test, row))
    names = []
    for i in tied:
        names.append("the " + test["names"][i])
    if len(tied) > 1:
        what = what + ". Before that morning %s share the highest estimate + bonus" % _n5_and(names)
        if values is not None:
            what = what + ", %s" % _n5_dec(_n5_hund(values[right]))
        what = what + ", and with the same numbers the game's mirror picks the %s" % test["names"][right]
    else:
        what = what + ". The highest estimate + bonus before that morning is the %s's" % test["names"][right]
        if values is not None:
            what = what + ", %s" % _n5_sum_text(board[right], values[right])
            if row in range(len(board)) and len(board[row]) > 0:
                what = what + ", and the %s's is %s" % (test["names"][row], _n5_sum_text(board[row], values[row]))
    if run["p_sum"] or run["p_read"]:
        return what, _N5_SLIP_HINTS["p"]
    slip = _n5_which_slip(fn, test, events)
    if slip is not None:
        return what, _N5_SLIP_HINTS[slip]
    if len(tied) > 1 and row in tied:
        if len(run["draws"]) == 0:
            return what + ". Your mirror drew no number from next_number", _N5_SLIP_HINTS["first"]
        return what, _N5_SLIP_HINTS["first"]
    return what, ("each morning after initialize, the mirror shows the row with the highest estimate + bonus on its "
                  "board: product = pick_highest(scores(rows), next_number), inside the loop.")


def _n5_explain_mornings(fn, run):
    """(what, hint) from the mornings that the mirror played, read from her answers; None when they are the game's."""
    test, events = run["test"], run["events"]
    k = len(test["p"])
    j = 0
    while j < len(events) and j < test["mornings"] and events[j][0] == int(test["shown"][j]) and events[j][1] == j:
        j = j + 1
    if j == len(events) and j == test["mornings"]:
        return None
    if j == len(events):
        return _n5_too_short(test, j)
    if j == test["mornings"]:
        return ("your mirror asked for her answer %s in a season of %d mornings" % (_n5_count(len(events), "time"),
                                                                                    test["mornings"]),
                "one answer per morning: call tap once in each round of each loop.")
    if events[j][1] != j:
        return _n5_wrong_morning(test, events, j)
    if j < k:
        return ("on %s your mirror showed %s. Initialize shows each product once, from the top row down: %s"
                % (_n5_morning(j), _n5_product(test, events[j][0]), _n5_product(test, j)),
                "on the mornings of initialize, product = morning: row 0 on morning 0, row 1 on morning 1, and so on.")
    return _n5_wrong_choice(fn, test, run, j)


def _n5_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
                    "cell, then run this check again." % name)
    if _n5_has_gap(fn):
        print(not_finished)
        return
    for other in _N5_NEEDS[name]:
        if _n5_uses(fn, other) and not _n5_works(other):
            print("⏳ %s uses your %s from %s, which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                  "check again." % (name, other, _N5_TASKS[other], other, other))
            return
    failure = _n5_first_failure(name, fn)
    if failure is None:
        print("✅ " + _N5_SUCCESS[name])
        return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    if name == "bonus" or name == "scores":
        if name == "bonus":
            hint = _n5_hint_bonus(fn, failure)
        else:
            hint = _n5_hint_scores(fn, failure)
        if failure["error"] is not None:
            print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
        elif failure["answer"] is None:
            print("❌ %s gave no answer (None), expected %s. %s." % (failure["call"], _n5_text(failure["expected"]),
                                                                     failure["why"]))
        else:
            print("❌ %s gave %s, expected %s. %s." % (failure["call"], _n5_text(failure["answer"]),
                                                       _n5_text(failure["expected"]), failure["why"]))
        print("   💡 Hint: " + hint)
        return
    test = failure["test"]
    head = "%s, %s" % (test["call"], test["about"])
    if failure["error"] is not None:
        where = ""
        if failure["line"]:
            where = " on the line  %s" % failure["line"]
        print("❌ %s: your function stopped with an error%s. %s" % (head, where, failure["error"]))
        print("   💡 Hint: " + _n5_explain_error(failure))
        return
    problem = _n5_shape(failure)
    if problem is None:
        problem = _n5_answers(failure)
    if problem is None:
        problem = _n5_explain_mornings(fn, failure)
    if problem is None:
        problem = ("your board differs from the game's", "compare your code with the four steps above the cell, then "
                                                         "open the hints.")
    what, hint = problem
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint)


def check_bonus(fn):
    """Task A: tests bonus on eight rows, from an empty row to the vacuum's whole season."""
    try:
        _n5_check("bonus", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def check_scores(fn):
    """Task B: tests scores on seven boards, from a board of one row to whole seasons."""
    try:
        _n5_check("scores", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def check_ucb(fn):
    """Task C: plays ucb on eight seasons (both customers, both collections of section 3) with the game's stream of
    each season, and compares each board with the game's, morning by morning."""
    try:
        _n5_check("ucb", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def _n5_ready(name, fn, what):
    """True when fn is finished and passes its check; else prints one calm line and gives False."""
    if not callable(fn):
        print("⏳ %s needs your function itself, with no brackets after %s." % (what, name))
        return False
    if _n5_has_gap(fn):
        print("⏳ %s is not finished yet (%s), so there is nothing to show. Finish it, run its cell, then run this "
              "cell again." % (name, _N5_TASKS[name]))
        return False
    if globals().get(name) is fn and not _n5_works(name):
        print("⏳ %s does not work yet (%s). Get check_%s(%s) to ✅ first, then run this cell again." % (
            name, _N5_TASKS[name], name, name))
        return False
    return True


def play(mirror, *inputs):
    """Plays one season with your mirror and gives its board: play(ucb, cinderella, moods, next_number) is
    ucb(cinderella, moods, next_number). When the mirror is not finished yet, stops with an error or gives something
    that is not a board, play prints one calm line and gives None, so the cells below still run."""
    import contextlib, io
    try:
        if not callable(mirror):
            print("⏳ play needs your function itself, as in play(ucb, cinderella, moods, next_number).")
            return None
        name = getattr(mirror, "__name__", "your mirror")
        if _n5_has_gap(mirror):
            task = ""
            if name in _N5_TASKS:
                task = " (%s)" % _N5_TASKS[name]
            print("⏳ %s is not finished yet%s, so there is no season to show. Finish it, run its cell, then run this "
                  "cell again." % (name, task))
            return None
        for other in _N5_TASKS:
            fn = globals().get(other)
            if other == name or not _n5_uses(mirror, other):
                continue
            if callable(fn) and _n5_has_gap(fn):
                print("⏳ %s uses your %s (%s), which is not finished yet. Finish it, run its cell, then run this cell "
                      "again." % (name, other, _N5_TASKS[other]))
                return None
            if not _n5_works(other):
                print("⏳ %s uses your %s (%s), which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                      "cell again." % (name, other, _N5_TASKS[other], other, other))
                return None
        copies = []
        for x in inputs:
            if isinstance(x, list):
                copies.append(list(x))
            else:
                copies.append(x)
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                answer = mirror(*copies)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check above says more." % (name, type(error).__name__,
                                                                                     error))
            return None
        rows = None
        if len(copies) > 0 and isinstance(copies[0], list):
            rows = len(copies[0])
        if not _n5_is_board(answer, rows):
            print("❌ %s gave %s, which is not a board. Its check above says more." % (name, _n5_short(answer)))
            return None
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the two folded cells at the top again, then this cell." % error)
        return None


def show_bonus(fn, dots):
    """Prints the bonus that your bonus gives a row with each number of dots in the list dots, with a bar as long as
    the bonus: the game's plot of 1 / square root of n, on its side."""
    try:
        if not _n5_ready("bonus", fn, "show_bonus"):
            return
        print(" dots   bonus")
        for n in dots:
            row = []
            for i in range(n):
                row.append(0)
            value, error = _n5_call(fn, (row,))
            if error is not None or not _n5_is_number(value):
                print("%5d   ?  (your bonus gave %s)" % (n, error or _n5_short(value)))
                continue
            print("%5d   %-5s  %s" % (n, _n5_dec(value), "█" * int(20 * value + 0.5)))
    except Exception as error:
        print("❌ show_bonus could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def show_ucb_board(rows, next_number=None, width=30):
    """Prints UCB's board as in the game: estimate + bonus on the far left, the estimate p̂, the bonus, the product,
    then one dot per showing, 🟠 a tap and ⚪ no tap. A ⭐ marks the highest estimate + bonus. With next_number, it
    also says which product the mirror shows next: pick_highest(scores(rows), next_number), a tie broken at random.
    The columns bonus and estimate + bonus come from your bonus (task A) and scores (task B) once they work."""
    try:
        if not _n5_is_board(rows):
            print("⏳ No board to show yet: show_ucb_board needs one list of 1s and 0s per product. It got %s."
                  % _n5_short(rows))
            return
        has_bonus = _n5_works("bonus")
        values = _n5_scores_of(rows)
        if not has_bonus or values is None:
            missing = []
            if not has_bonus:
                missing.append("bonus (task A)")
            if not _n5_works("scores"):
                missing.append("scores (task B)")
            if missing:
                print("⏳ The columns ? fill in once your %s pass%s the check." % (_n5_and(missing),
                                                                                   "" if len(missing) > 1 else "es"))
        top = []
        if values is not None:
            best = max(values)
            for k in range(len(rows)):
                if abs(values[k] - best) <= 1e-9:
                    top.append(k)
        print("p̂ + bonus      p̂  bonus")
        for k in range(len(rows)):
            row = rows[k]
            s, e, b = "?", "?", "?"
            if len(row) > 0:
                e = _n5_dec(estimate(row))
                if has_bonus:
                    value, error = _n5_call(globals()["bonus"], (row,))
                    if error is None and _n5_is_number(value):
                        b = _n5_dec(value)
                if values is not None:
                    # the two numbers as the board shows them, added, so the column adds up on screen as in the game
                    s = _n5_dec(_n5_hund(estimate(row)) + _n5_hund(values[k] - estimate(row)))
            dots = ""
            for x in row[:width]:
                if x:
                    dots = dots + "🟠"
                else:
                    dots = dots + "⚪"
            if len(row) > width:
                rest = row[width:]
                dots = dots + " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
            name = "#%d" % k
            if len(rows) == len(_N5_EMOJI):
                name = _N5_EMOJI[k]
            star = ""
            if k in top:
                star = "  ⭐"
            print("%9s  %5s  %5s  %s  %s%s" % (s, e, b, name, dots, star))
        if next_number is None or values is None:
            return
        pick = pick_highest(values, next_number)
        if len(top) == 1:
            print("The highest estimate + bonus: %s, %s. The mirror shows it." % (_n5_label(pick),
                                                                                   _n5_sum_text(rows[pick],
                                                                                                values[pick])))
        else:
            print("%s products tie at %s. pick_highest(scores(rows), next_number) draws a number and picks %s."
                  % (_N5_WORDS.get(len(top), str(len(top))), _n5_dec(_n5_hund(values[pick])), _n5_label(pick)))
    except Exception as error:
        print("❌ show_ucb_board could not run (%s). Run the two folded cells at the top again, then this cell."
              % error)


print("✅ The checks of notebook 5 are ready.")

## The strategy in four steps

The game told UCB in four steps:

1. **Initialize**: show each product once: its first estimate and its first bonus.
2. **Show the highest**: the product with the highest estimate + bonus. A tie is broken at random.
3. **Update**: tap or no tap, its estimate and its bonus change.
4. **Repeat**: every morning, to the end of the season.

The board has two columns more than in the earlier sections: the **bonus**, between the estimate and the product,
and **estimate + bonus** on the far left. Here are the two customers, with the probability that each of them taps
each product, and the moods of the race season, the season of every comparison in the game.

In [ ]:
# The probability that each customer taps each product, in board order (invented for the game).
# Only tap reads them: the mirror itself sees only the dots of its board.
cinderella = [0.6, 0.2, 0.2, 0.4, 0.2]
rapunzel = [0.4, 0.8, 0.2, 0.2, 0.2]

# The race season of the game: her mood on each of its mornings
moods = season_moods(RACE)

for i in range(len(NAMES)):
    print("row", i, "   Cinderella", cinderella[i], "   Rapunzel", rapunzel[i], "   ", EMOJI[i], NAMES[i])

## 🎯 Task A. The exploration bonus

The bonus favours the products the mirror has not explored enough. A row with n dots gets the bonus 1 / √n, so a
row with one dot gets 1 and a row with 9 dots gets 1 / 3. The kit gives the square root, `sqrt`: `sqrt(9)` is
`3.0`.

Write `bonus(row)`. It gets the row of one product, a list of 1s and 0s, and returns its bonus. As with
`estimate`, a product that was never shown has an empty row and no bonus yet: the first two lines return `None`.
You fill in the rest, in two steps:

1. **Count the dots**: n is the number of showings, one per dot, tap or no tap: `len(row)`.
2. **The bonus**: 1 divided by the square root of n.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# n counts every dot of the row, taps and no taps alike: len(row).
# sqrt(showings) is the square root of showings, and the bonus is 1 divided by it.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     showings = len(row)
#     return 1 / ...          # the square root of showings

In [ ]:
# 🎯 YOUR TURN, task A: the exploration bonus of one product
def bonus(row):
    # row: the dots of one product, 1 for a tap and 0 for no tap.
    # Returns its bonus, 1 / square root of the number of dots. None when the product was never shown.
    if len(row) == 0:
        return None             # never shown: no bonus yet (the game writes ?)
    showings = ...              # 🎯 YOUR TURN: the number of dots in the row
    return ...                  # 🎯 YOUR TURN: 1 divided by the square root of showings (use sqrt)

In [ ]:
check_bonus(bonus)

## How the bonus shrinks

`show_bonus(bonus, dots)` gives your `bonus` a row with each number of dots in the list and prints the bonus with
a bar as long as it: the game's plot of 1 / √n, on its side.

💭 **Think first.** Which drop is bigger: from 1 dot to 2, or from 50 dots to 100?

In [ ]:
# The bonus of a row with 1 dot, 2 dots, and so on, up to a whole season of 100 mornings
show_bonus(bonus, [1, 2, 3, 4, 5, 10, 20, 50, 100])

**Reading the result.** 1 dot gives 1, 2 dots 0.71, 3 dots 0.58, 4 dots 0.5. The more dots, the smaller the bonus. It
drops fast over the first dots, then slowly: from 1 dot to 2 it loses 0.29, from 50 dots to 100 only 0.04. A product
shown on all 100 mornings of a season still has a bonus of 0.1.

## 🎯 Task B. Estimate + bonus

Each morning the mirror adds the two columns of its board, row by row: estimate + bonus, the column on the far
left of the game's board. Write `scores(rows)`. It gets a board in which every row has at least one dot, as after
initialize, and returns estimate + bonus for every row, in board order. The kit's `estimate(row)` gives the
estimate and your `bonus(row)` the bonus.

On the race season, after initialize, only the mattress got a tap. So `scores` gives `[1, 1, 1, 2, 1]` for that board: 0 + 1
for the four products with no tap, 1 + 1 for the mattress.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop for row in rows visits the rows of the board one by one, from the vacuum's down.
# For each row: score = estimate(row) + bonus(row), then values.append(score) puts it at the end of the list.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for row in rows:
#         score = estimate(row) + ...       # the bonus of the row, from task A
#         values.append(score)

In [ ]:
# 🎯 YOUR TURN, task B: estimate + bonus for every product on the board
def scores(rows):
    # rows: the board, one list of 1s and 0s per product. Every row has at least one dot (after initialize).
    # Returns estimate + bonus for every row, in board order.
    values = []
    for row in rows:
        score = ...             # 🎯 YOUR TURN: the estimate of the row plus its bonus
        ...                     # 🎯 YOUR TURN: put score at the end of the list values
    return values

In [ ]:
check_scores(scores)

## The board of the game

`show_ucb_board(rows, next_number)` prints UCB's board as in the game: estimate + bonus on the far left, then the
estimate p̂, the bonus, the product and its dots. A ⭐ marks the highest estimate + bonus. With `next_number` it
also says which product the mirror shows next, `pick_highest(scores(rows), next_number)`: the kit's
`pick_highest` gives the row with the highest value, and breaks a tie at random with a number from `next_number`,
exactly as the game does. `strategy_numbers(RACE, "ucb")` is the mirror's own stream of numbers in the race season.

The game counts the mornings from 1: initialize takes mornings 1 to 5, and morning 6 is the first one after
it.

💭 **Think first.** After initialize, only the mattress has a tap. Which product does the mirror show on morning
6? And if Cinderella does not tap it, on morning 7?

In [ ]:
# The race season at Cinderella's: the game's board after initialize (mornings 1 to 5) and after 7 mornings
after_initialize = [[0], [0], [0], [1], [0]]
after_7_mornings = [[0], [0], [0], [1, 0, 0], [0]]
next_number = strategy_numbers(RACE, "ucb")     # the mirror's own stream, for a tie

show_ucb_board(after_initialize, next_number)
print()
show_ucb_board(after_7_mornings, next_number)

**Reading the result.** After initialize every row has one dot, so every bonus is 1/√1 = 1 and every estimate is
0 or 1. The mattress scores 1 + 1 = 2 and the others 0 + 1 = 1: on morning 6 the mirror shows the mattress.
No tap: 0.5 + 0.71 = 1.21, still the highest, so morning 7 shows it again. No tap again: 0.33 + 0.58 = 0.91. On morning 8 four products tie
at 1, and the draw picks the vacuum, as in the game. The mirror does not know it yet, but the vacuum is Cinderella's
favourite.

## 🎯 Task C. UCB

Write `ucb(p, moods, next_number)`. It plays a whole season and returns its board, in the game's four steps.
Python counts the mornings from 0, as it counts the rows, so the game's morning 6 is `moods[5]`.

1. **Initialize**: on mornings 0 to `len(p) - 1`, the product of the same number: row 0 on morning 0, row 1 on
   morning 1, and so on. Then every row has an estimate and a bonus.
2. **Show the highest**: `pick_highest(scores(rows), next_number)` gives the row with the highest estimate +
   bonus, a tie broken at random.
3. **Update**: her answer, `tap(p[product], moods[morning])`, goes at the end of that product's row. Its estimate
   and its bonus change.
4. **Repeat**: steps 2 and 3 on every remaining morning, `range(len(p), len(moods))`.

With the stream `strategy_numbers(RACE, "ucb")`, your mirror plays the game's race season, morning by morning.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Initialize: on morning m the mirror shows product m, so product = morning.
# Each remaining morning: scores(rows) gives estimate + bonus for every row, and pick_highest(values,
# next_number) the row with the highest one. Pick inside the loop: every new dot changes the board.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# Her answer to the product shown is tap(p[product], moods[morning]): 1 for a tap, 0 for no tap.
# rows[product].append(dot) adds it at the end of that product's row: that is the update.

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     for morning in range(len(p), len(moods)):
#         product = pick_highest(scores(rows), next_number)
#         dot = tap(p[product], moods[morning])
#         rows[product].append(dot)

In [ ]:
# 🎯 YOUR TURN, task C: UCB
def ucb(p, moods, next_number):
    # p: the probability of a tap for each product, in board order. Only tap reads it.
    # moods: her mood on each morning of the season. next_number: the mirror's own stream of numbers, for a tie.
    # Returns the board of the whole season.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product

    # 1. Initialize: each product once, from the top row down
    for morning in range(len(p)):
        product = ...                           # 🎯 YOUR TURN: the product of this morning
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row

    # 2, 3 and 4: on every remaining morning, show the highest estimate + bonus, then update
    for morning in range(len(p), len(moods)):
        product = ...                           # 🎯 YOUR TURN: the highest estimate + bonus (pick_highest)
        dot = ...                               # 🎯 YOUR TURN: her answer to that product this morning
        ...                                     # 🎯 YOUR TURN: update: add dot at the end of its row
    return rows

In [ ]:
check_ucb(ucb)

## Cinderella's season

`play(ucb, cinderella, moods, next_number)` runs `ucb(cinderella, moods, next_number)` and gives its board. While
the function is not finished, it prints one calm line and gives `None`.

💭 **Think first.** A product left alone keeps its bonus, while the bonus of the product shown shrinks. How many
of the 100 mornings do you expect the vacuum to get? And the three products that Cinderella taps with probability 0.2?

In [ ]:
# UCB at Cinderella's on the race season
ucb_rows = play(ucb, cinderella, moods, strategy_numbers(RACE, "ucb"))
show_ucb_board(ucb_rows)
print("UCB's rate, taps / mornings:", rate(ucb_rows))

**Reading the result.** 52 taps in 100 mornings, a rate of 0.52: the game's numbers. After 100 mornings: 78 dots for
the vacuum, 13 for the mattress, 3 for each of the other three. Their bonus is still 0.58, and their estimate is 0:
0 + 0.58 stays below the vacuum's 0.6 + 0.11 = 0.71. The board shows the first 30 dots of the vacuum's row and counts the other 48.

## All the mirrors

The game's last scene, **All the mirrors**, races every mirror at Rapunzel's on the race season, all on the same
moods: the Queen's mirror, Random, estimate then exploit, epsilon-greedy and UCB. Her favourite, the ladder 🪜
(p = 0.8), stands far above her other products: the next best, the vacuum 🤖, has p = 0.4. The folded cell below has
the four mirrors of notebooks 2 to 4, as in their solutions. Estimate then exploit uses the game's n = 5 and
epsilon-greedy the game's eps = 0.2. Run it.

In [ ]:
#@title From notebooks 2 to 4: the Queen's mirror, Random, estimate then exploit, epsilon-greedy (run me) { display-mode: "form" }
# The mirrors of notebooks 2 to 4, as in their solutions. Run this cell; open it if you like.

def queen_mirror(p, moods):
    # The Queen's mirror knows every probability: every morning it shows the product with the highest one.
    best = 0
    for i in range(len(p)):
        if p[i] > p[best]:      # product i beats the best so far
            best = i
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        dot = tap(p[best], moods[morning])    # 1 if she taps the best product this morning, else 0
        rows[best].append(dot)                # one more dot in its row
    return rows


def random_mirror(p, moods, next_number):
    # Random draws a card from the hat each morning: every product has the same chance.
    rows = []
    for i in range(len(p)):
        rows.append([])         # one empty row per product
    for morning in range(len(moods)):
        card = int(next_number() * len(p))    # a card from the hat: a row from 0 to len(p) - 1
        dot = tap(p[card], moods[morning])    # her answer this morning: 1 or 0
        rows[card].append(dot)                # one more dot in the card's row
    return rows


def round_robin(p, moods, n):
    # The test of estimate then exploit: n rounds, every product once per round, from the top row down.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product
    test = n * len(p)                           # n rounds of len(p) mornings
    for morning in range(test):
        product = morning % len(p)              # in turn: 0, 1, 2, 3, 4, then 0 again
        dot = tap(p[product], moods[morning])   # her answer: 1 for a tap, 0 for no tap
        rows[product].append(dot)               # one more dot in that product's row
    return rows


def estimate_then_exploit(p, moods, next_number, n):
    # Estimate then exploit: the test, then the highest estimate on every remaining morning.
    rows = round_robin(p, moods, n)             # 1. initialize and test: n rounds
    test = n * len(p)
    winner = pick_highest(estimates(rows), next_number)    # 2. the highest estimate, picked once
    for morning in range(test, len(moods)):     # 3. exploit: the winner on every remaining morning
        dot = tap(p[winner], moods[morning])
        rows[winner].append(dot)
    return rows


def spin_the_roulette(rows, next_number, eps):
    # The roulette of epsilon-greedy: on a share eps of the spins a card from the hat, otherwise the highest estimate.
    spin = next_number()                        # where the needle stops: a number between 0 and 1
    if spin < eps:                              # 🎩 the small slice
        product = int(next_number() * len(rows))                # a card from the hat: any row at random
    else:                                       # 🥇 the large slice
        product = pick_highest(estimates(rows), next_number)    # the highest estimate, a tie at random
    return product


def epsilon_greedy(p, moods, next_number, eps):
    # Epsilon-greedy: initialize, then spin the roulette every morning.
    rows = []
    for i in range(len(p)):
        rows.append([])                         # one empty row per product
    for morning in range(len(p)):               # 1. Initialize: each product once, from the top row down
        product = morning
        dot = tap(p[product], moods[morning])
        rows[product].append(dot)
    for morning in range(len(p), len(moods)):
        product = spin_the_roulette(rows, next_number, eps)    # 2. spin: 🥇 or 🎩
        dot = tap(p[product], moods[morning])   # 3. sample: her answer, 1 or 0
        rows[product].append(dot)               # 4. update: the next spin sees the new estimate
    return rows


print("✅ queen_mirror, random_mirror, estimate_then_exploit and epsilon_greedy from notebooks 2 to 4 are ready.")

💭 **Think first.** The Queen's mirror shows the ladder on all 100 mornings. On how many do you expect UCB to show
it? Which of the three mirrors that learn comes closest to the Queen's mirror?

In [ ]:
# The last race: every mirror at Rapunzel's on the race season, all on the same moods
n = 5          # estimate then exploit's n in the game
eps = 0.2      # epsilon-greedy's eps in the game

queen_rows = play(queen_mirror, rapunzel, moods)
ucb_rows = play(ucb, rapunzel, moods, strategy_numbers(RACE, "ucb"))
eg_rows = play(epsilon_greedy, rapunzel, moods, strategy_numbers(RACE, "epsilon-greedy"), eps)
ete_rows = play(estimate_then_exploit, rapunzel, moods, strategy_numbers(RACE, "estimate then exploit"), n)
random_rows = play(random_mirror, rapunzel, moods, strategy_numbers(RACE, "random"))
print("The Queen's mirror:", rate(queen_rows))
print("UCB:", rate(ucb_rows))
print("Epsilon-greedy, eps =", eps, ":", rate(eg_rows))
print("Estimate then exploit, n =", n, ":", rate(ete_rows))
print("Random:", rate(random_rows))

**Reading the result.** The Queen's mirror 0.78, UCB 0.7, epsilon-greedy 0.67, estimate then exploit 0.62, Random 0.32:
the rates at morning 100 on the game's chart of the last race. On the board of each mirror that learns, the
ladder's row has the most dots: 88 for UCB, 84 for epsilon-greedy, 80 for estimate then exploit. Random showed the
ladder on 15 mornings. `len(ucb_rows[1])` counts the dots of the ladder's row. One season is one roll of luck, so the
next cells play many.

## Many seasons

`average_rates(p, n, eps, first_seed, seasons)` plays every mirror for many seasons, one season per seed, and
prints the rate of all those mornings. In each season every mirror meets the same moods. The two cells after it
play the seasons at Rapunzel's, then at Cinderella's; each takes a few seconds.

In [ ]:
# Given: the taps of a board, and every mirror over many seasons
def count_taps(rows):
    # All the 1s on the board, added up
    taps = 0
    for row in rows:
        for dot in row:
            taps = taps + dot
    return taps


def average_rates(p, n, eps, first_seed, seasons):
    # Plays every mirror for many seasons, one season per seed, and prints the rate of all those mornings.
    # In each season every mirror meets the same moods.
    queen_taps = 0
    ucb_taps = 0
    eg_taps = 0
    ete_taps = 0
    random_taps = 0
    for season in range(seasons):
        seed = first_seed + season
        moods = season_moods(seed)
        rows = play(ucb, p, moods, strategy_numbers(seed, "ucb"))
        if rows == None:
            return              # ucb is not ready yet: play has said why
        ucb_taps = ucb_taps + count_taps(rows)
        queen_taps = queen_taps + count_taps(play(queen_mirror, p, moods))
        rows = play(epsilon_greedy, p, moods, strategy_numbers(seed, "epsilon-greedy"), eps)
        eg_taps = eg_taps + count_taps(rows)
        rows = play(estimate_then_exploit, p, moods, strategy_numbers(seed, "estimate then exploit"), n)
        ete_taps = ete_taps + count_taps(rows)
        random_taps = random_taps + count_taps(play(random_mirror, p, moods, strategy_numbers(seed, "random")))
    mornings = seasons * SEASON
    print("Seasons:", seasons, "   first seed:", first_seed, "   mornings in all:", mornings)
    print("The Queen's mirror:", queen_taps / mornings)
    print("UCB:", ucb_taps / mornings)
    print("Epsilon-greedy, eps =", eps, ":", eg_taps / mornings)
    print("Estimate then exploit, n =", n, ":", ete_taps / mornings)
    print("Random:", random_taps / mornings)

In [ ]:
# 1,000 seasons at Rapunzel's: seeds 1 to 1000, every mirror on the same moods in each
average_rates(rapunzel, n, eps, 1, 1000)

In [ ]:
# The same 1,000 seasons at Cinderella's
average_rates(cinderella, n, eps, 1, 1000)

**Reading the result.** Over 1,000 seasons at Rapunzel's: the Queen's mirror about 0.8, UCB about 0.72, epsilon-greedy
about 0.66, estimate then exploit about 0.65 and Random about 0.36. At Cinderella's: the Queen's mirror about 0.6, UCB about
0.51, epsilon-greedy about 0.47, estimate then exploit about 0.46 and Random about 0.32. At both customers the mirrors
finish in the order of the last race, and UCB comes closest to the Queen's mirror without knowing p.

## The end of the bandits

That is notebook 5, the last one. Over five notebooks you computed the estimate from the dots and wrote four
mirrors, Random, estimate then exploit, epsilon-greedy and UCB, each raced against the Queen's mirror. The three
that learn all start by initializing, then share the mornings between exploring and exploiting in their own way:
a test of n rounds, a roulette, or a bonus for the products with few dots.

Back in the game, the scene **All the mirrors** races them at Rapunzel's on other seasons too: press
**Another season**.